Assignment 2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import glob

In [ ]:
def load(ticker):
    files = sorted(glob.glob(f"data/{ticker}_md_*.csv.gz"))
    return pd.concat([pd.read_csv(f, index_col=0) for f in files], ignore_index=True)

def build(df):
    mid = ((df["BP1"] + df["SP1"]) / 2).where((df["BP1"] > 0) & (df["SP1"] > 0))
    df["mid"] = mid.groupby(df["date"]).ffill()
    df["prev_mid"] = df.groupby("date")["mid"].shift(1)
    tr = df[df["lastPx"].notna()].dropna(subset=["prev_mid"]).copy()
    tr["x"] = np.sign(tr["lastPx"] - tr["prev_mid"])
    tr["r"] = tr.groupby("date")["prev_mid"].shift(-1) / tr["prev_mid"] - 1
    return tr.dropna(subset=["r"])[["date", "time", "r", "x"]]

def add_lags(tr, maxlag):
    g = tr.groupby("date")
    for k in range(1, maxlag + 1):
        tr[f"r_{k}"] = g["r"].shift(k)
        tr[f"x_{k}"] = g["x"].shift(k)
    return tr.dropna()

def fit_hasbrouck(tr, l=10, m=10, n=10, o=10):
    d = add_lags(tr.copy(), max(l, m, n, o))
    X1 = d[[f"r_{k}" for k in range(1, l + 1)] + ["x"] + [f"x_{k}" for k in range(1, m + 1)]]
    X2 = d[[f"r_{k}" for k in range(1, n + 1)] + [f"x_{k}" for k in range(1, o + 1)]]
    ret = sm.OLS(d["r"], sm.add_constant(X1)).fit()  # eq (1)
    trd = sm.OLS(d["x"], sm.add_constant(X2)).fit()  # eq (2)
    return ret, trd

def coef_table(ret, trd):
    # map statsmodels names to Hasbrouck names
    def split(params, r_name, x_name):
        out = {r_name: {}, x_name: {}}
        for name, v in params.drop("const").items():
            var, _, k = name.partition("_")
            out[r_name if var == "r" else x_name][int(k or 0)] = v
        return out
    t = pd.DataFrame({**split(ret.params, "a", "c"), **split(trd.params, "b", "d")})
    return t.sort_index()[["a", "b", "c", "d"]].rename_axis("k")

In [ ]:
returns, trades = fit_hasbrouck(build(load("0050")))
coef_table(returns, trades)

In [ ]:
returns, trades = fit_hasbrouck(build(load("0056")))
coef_table(returns, trades)

In [ ]:
returns, trades = fit_hasbrouck(build(load("2330")))
coef_table(returns, trades)